In [ ]:

# MASTER DATA WRANGLING PIPE

# - Ingests multiple CSVs with different schemas
# - Normalizes to a single master schema
# - Enriches with: job_basket, location_bucket, seniority_level, skills, salary
# - Filters to data roles (toggle)
# - De-dupes and exports a clean master CSV
#
# Requirements: pandas, numpy
#

import pandas as pd
import numpy as np
import re, ast
from pathlib import Path

# ---------- CONFIG ----------
# columns in the final resulting dataset
MASTER_COLS = [
    'title','company_name','job_location','via','description','extensions',
    'description_tokens','SQL','Python','Excel','Tableau','Power BI','Cloud',
    'skill_buckets','skill_buckets_str','job_basket','listed_time',
    'location_bucket','seniority_level',
    'salary_min_annual','salary_max_annual','salary_currency','salary_mid_annual'
]
# if any ingested file doest have any of these columm it will be given these default values
DEFAULTS = {
    'title':'', 'company_name':'', 'job_location':'', 'via':'',
    'description':'', 'extensions':'',
    'description_tokens': [],
    'SQL': False, 'Python': False, 'Excel': False, 'Tableau': False, 'Power BI': False, 'Cloud': False,
    'skill_buckets': [], 'skill_buckets_str':'',
    'job_basket':'Unspecified',
    'listed_time': pd.NA,
    'location_bucket':'Other',
    'seniority_level':'Unspecified',
    'salary_min_annual': np.nan, 'salary_max_annual': np.nan,
    'salary_currency': pd.NA, 'salary_mid_annual': np.nan
}
# ensuring the datatypes for each columns is consistent
DTYPE_CASTS = {
    'title':'string', 'company_name':'string', 'job_location':'string', 'via':'string',
    'description':'string', 'extensions':'string',
    'SQL':'boolean','Python':'boolean','Excel':'boolean','Tableau':'boolean','Power BI':'boolean','Cloud':'boolean',
    'job_basket':'string','location_bucket':'string','seniority_level':'string','salary_currency':'string'
}

# Toggle: keep only data roles (excludes "Other")
ONLY_DATA_ROLES = True

# ---------- HELPERS ----------
def parse_listlike(x):
    """description_tokens / skill_buckets -> list (handle stringified lists)."""
    if isinstance(x, list): return x
    if pd.isna(x): return []
    s = str(x).strip()
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list): return v
    except Exception:
        pass
    if s.startswith('[') and s.endswith(']'):
        s = s[1:-1]
    parts = [p.strip().strip("'").strip('"') for p in s.split(',') if p.strip()]
    return parts

def coerce_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure df has all MASTER_COLS with defaults + types + order."""
    df = df.copy()

    # add missing cols (use per-row list for list-typed defaults)
    for c in MASTER_COLS:
        if c not in df.columns:
            default = DEFAULTS.get(c, np.nan)
            if isinstance(default, list):
                df[c] = [[] for _ in range(len(df))]
            else:
                df[c] = default

    # type casts (best effort)
    for c, dt in DTYPE_CASTS.items():
        if c in df.columns:
            try:
                df[c] = df[c].astype(dt)
            except Exception:
                if dt == 'boolean':
                    df[c] = df[c].astype('boolean')

    # list-like normalization
    df['description_tokens'] = df['description_tokens'].map(parse_listlike)
    if 'skill_buckets' in df.columns:
        df['skill_buckets'] = df['skill_buckets'].map(parse_listlike)
    else:
        df['skill_buckets'] = [[] for _ in range(len(df))]
    df['skill_buckets_str'] = df['skill_buckets'].apply(
        lambda L: ', '.join(L) if isinstance(L, list) else ''
    )

    # final order
    return df[MASTER_COLS]

# ---------- NORMALIZERS PER SOURCE SHAPE ----------
def normalize_gsearch_q(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """Normalize df1/df2/df3 (gsearch 2023 q-files)."""
    df = df.copy()
    # drop junk unnamed
    df = df.loc[:, ~df.columns.str.contains(r'^Unnamed')]
    # ensure extensions exists
    if 'extensions' not in df.columns: df['extensions'] = ''
    # listed_time from posted_at or date_time → year
    lt = pd.to_datetime(df.get('posted_at', pd.NaT), errors='coerce')
    if lt.isna().all():
        lt = pd.to_datetime(df.get('date_time', pd.NaT), errors='coerce')
    df['listed_time'] = lt.dt.year.astype('Int64')

    out = coerce_schema(df)
    out['source_file'] = source_name
    out['listed_year'] = pd.to_numeric(out['listed_time'], errors='coerce').astype('Int64')
    return out

def normalize_2024(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """Normalize 2024 schema to master."""
    df = df.copy()
    # rename to master
    df = df.rename(columns={
        'location': 'job_location',
        'skills_desc': 'description_tokens',
        'posting_domain': 'via'
    })

    # build extensions row-wise from a few useful hints
    ext_cols = [c for c in ['formatted_work_type', 'pay_period', 'compensation_type'] if c in df.columns]
    if ext_cols:
        df[ext_cols] = df[ext_cols].astype(str).replace('nan', '')
        df['extensions'] = df[ext_cols].agg(' | '.join, axis=1).str.strip(' |')
    else:
        df['extensions'] = ''

    # listed_time → year
    if 'listed_time' in df.columns:
        yr = pd.to_datetime(df['listed_time'], errors='coerce').dt.year.astype('Int64')
    else:
        yr = pd.to_numeric(df.get('year', pd.NA), errors='coerce').astype('Int64')
    df['listed_time'] = yr

    # tokens
    if 'description_tokens' not in df.columns:
        df['description_tokens'] = df.get('skills_desc', pd.NA)
    df['description_tokens'] = df['description_tokens'].map(parse_listlike)

    out = coerce_schema(df)
    out['source_file'] = source_name
    out['listed_year'] = pd.to_numeric(out['listed_time'], errors='coerce').astype('Int64')
    return out

# ---------- ENRICHERS (VECTORIZED) ----------
def add_job_basket(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    title_s = df['title'].fillna('').str.lower()
    rx = {
        "Data Analytics": r"\b(?:data analyst|analytics|marketing analyst|pricing analyst)\b",
        "Data Science": r"\b(?:data scientist|research scientist|quant|ai researcher)\b",
        "Data Engineer": r"\b(?:data engineer|etl engineer|pipeline|sql developer)\b",
        "Business Intelligence Analyst": r"\b(?:bi analyst|business intelligence|reporting analyst)\b",
        "Machine Learning Engineer": r"\b(?:ml engineer|machine learning|deep learning|ai engineer)\b"
    }
    conds = [title_s.str.contains(p, regex=True, na=False) for p in rx.values()]
    choices = list(rx.keys())
    df['job_basket'] = np.select(conds, choices, default='Other')
    return df

def add_location_bucket(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    loc_s  = df['job_location'].fillna('').str.lower()
    desc_s = df['description'].fillna('').str.lower()
    ext_s  = df['extensions'].fillna('').astype(str).str.lower()
    all_s  = loc_s + ' ' + desc_s + ' ' + ext_s

    cond_remote = all_s.str.contains(r'\b(?:remote|anywhere|work from home|telecommute)\b', regex=True)
    cond_us = all_s.str.contains(
        r'\b(?:united states|usa|u\.s\.|us\b|al|ak|az|ar|ca|co|ct|de|fl|ga|hi|ia|id|il|in|ks|ky|la|ma|md|me|mi|mn|mo|ms|mt|nc|nd|ne|nh|nj|nm|nv|ny|oh|ok|or|pa|ri|sc|sd|tn|tx|ut|va|vt|wa|wi|wv)\b',
        regex=True
    )
    cond_can = all_s.str.contains(
        r'\b(?:canada|toronto|vancouver|ontario|bc|british columbia|quebec|qc|alberta|ab|manitoba|mb|saskatchewan|sk|nova scotia|ns|calgary|montreal|ottawa)\b',
        regex=True
    )
    df['location_bucket'] = np.select(
        [cond_remote, cond_us, cond_can],
        ['Remote', 'United States', 'Canada'],
        default='Other'
    )
    return df

def add_seniority(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    s = (df['title'].fillna('') + ' ' + df['description'].fillna('')).str.lower()
    rules = [
        (r'\b(?:vp|vice president|chief|cto|cdo|cdao)\b',           'Executive'),
        (r'\b(?:director|head of|sr\.?\s*manager|senior manager)\b','Director+'),
        (r'\b(?:manager|lead(?:er)? of|people manager)\b',          'Manager'),
        (r'\b(?:principal|staff|lead|architect)\b',                 'Lead/Principal'),
        (r'\b(?:senior|sr\.?|level\s*iii|iii)\b',                   'Senior'),
        (r'\b(?:mid|intermediate|level\s*ii|ii)\b',                 'Mid'),
        (r'\b(?:junior|jr\.?|entry[-\s]*level|level\s*i|i|associate)\b','Junior/Entry'),
        (r'\b(?:intern|internship|co[-\s]?op|coop)\b',              'Intern'),
    ]
    conds = [s.str.contains(rx, regex=True, na=False) for rx,_ in rules]
    labels = [lab for _,lab in rules]
    df['seniority_level'] = np.select(conds, labels, default='Unspecified')
    return df

def extract_skills(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # -------- unified text --------
    title_s = df.get('title', '').fillna('').astype(str)
    desc_s  = df.get('description', '').fillna('').astype(str)

    tokens_col = df.get('description_tokens')
    if tokens_col is not None:
        tokens_s = tokens_col.apply(lambda x: ' '.join(x) if isinstance(x, list)
                                              else ('' if pd.isna(x) else str(x)))
    else:
        tokens_s = pd.Series([''] * len(df), index=df.index)

    text_s = (title_s + ' ' + desc_s + ' ' + tokens_s).str.lower()

    # -------- regex dictionaries --------
    # Cloud sub-skills (expanded)
    cloud_rx = {
        # Providers
        'aws'       : r'\b(?:aws|amazon web services)\b|(?<!\w)(?:s3|glue|athena|redshift)\b',
        'azure'     : r'\b(?:azure)\b|(?<!\w)(?:synapse|data\s*factory|adf|databricks\s*on\s*azure)\b',
        'gcp'       : r'\b(?:gcp|google cloud)\b|(?<!\w)(?:bigquery|dataflow|pub/sub|pubsub)\b',

        # Data platforms / engines
        'snowflake' : r'\b(?:snowflake)\b',
        'redshift'  : r'\b(?:redshift)\b',
        'bigquery'  : r'\b(?:bigquery)\b',
        'databricks': r'\b(?:databricks)\b',
        'spark'     : r'\b(?:apache\s+spark|pyspark|spark\s*sql)\b',

        # Orchestration / ELT / Streaming
        'airflow'   : r'\b(?:airflow|apache\s*airflow)\b',
        'dbt'       : r'\b(?:dbt|data build tool)\b',
        'kafka'     : r'\b(?:kafka|apache\s*kafka)\b',

        # Infra (common in cloud data stacks)
        'docker'    : r'\b(?:docker)\b',
        'kubernetes': r'\b(?:kubernetes|k8s)\b',
        'terraform' : r'\b(?:terraform|iac|infrastructure as code)\b',
    }

    # Top-level buckets (Cloud is derived from cloud_rx below)
    rx = {
        'SQL'     : r'\b(?:sql|mysql|postgres(?:ql)?|tsql|oracle\s+sql|snowflake)\b',
        'Python'  : r'\b(?:python|pandas|numpy|scikit-?learn|sklearn)\b',
        'Excel'   : r'\b(?:excel|v\s*look\s*up|vlookup|pivot(?:\s*table)?s?)\b',
        'Tableau' : r'\b(?:tableau)\b',
        'Power BI': r'\b(?:power\s*-?\s*bi|pbi)\b',
    }

    # -------- vectorized matches --------
    skill_df = pd.DataFrame(
        {k: text_s.str.contains(pat, regex=True, na=False) for k, pat in rx.items()},
        index=df.index
    )

    cloud_df = pd.DataFrame(
        {k: text_s.str.contains(pat, regex=True, na=False) for k, pat in cloud_rx.items()},
        index=df.index
    )

    # Derive Cloud + nested list
    skill_df['Cloud'] = cloud_df.any(axis=1)
    skill_df['Cloud_list'] = cloud_df.apply(lambda r: [k for k, v in r.items() if v], axis=1)

    # -------- join + lists --------
    to_drop = [
        'SQL','Python','Excel','Tableau','Power BI','Cloud','Cloud_list',
        'skill_buckets','skill_buckets_str','skill_details','skill_details_str','skills_flat'
    ]
    df = df.drop(columns=[c for c in to_drop if c in df.columns], errors='ignore')
    df = df.join(skill_df)

    bucket_cols = ['SQL','Python','Excel','Tableau','Power BI','Cloud']
    present_mask = df[bucket_cols].fillna(False)
    long = present_mask.stack()
    long = long[long].rename('present').reset_index()
    skills_list = long.groupby('level_0')['level_1'].agg(list)

    empty_lists = pd.Series([[]] * len(df), index=df.index)
    skills_list = skills_list.reindex(df.index).combine_first(empty_lists)

    df['skill_buckets'] = skills_list
    df['skill_buckets_str'] = df['skill_buckets'].apply(lambda L: ', '.join(L))

    # Nested details
    cloud_lists = df['Cloud_list'].apply(lambda L: L if isinstance(L, list) else [])
    df['skill_details'] = cloud_lists.apply(lambda L: {'Cloud': L} if L else {})

    def pretty_details(d: dict) -> str:
        if not d: return ''
        parts = []
        for k, v in d.items():
            parts.append(f"{k}:[{'; '.join(v)}]" if isinstance(v, list) and v else f"{k}:[]")
        return ', '.join(parts)

    df['skill_details_str'] = df['skill_details'].apply(pretty_details)

    # Optional: flattened list for counting
    def flatten_skills(i):
        buckets = df.at[i, 'skill_buckets'] if isinstance(df.at[i, 'skill_buckets'], list) else []
        clouds  = df.at[i, 'Cloud_list'] if isinstance(df.at[i, 'Cloud_list'], list) else []
        return list(buckets) + [f"Cloud:{s}" for s in clouds]

    df['skills_flat'] = pd.Index(df.index).map(flatten_skills)
    return df

    # # Add nested details for Cloud (dict + pretty string) update
    # def pretty_details(d):
    #  if not d: 
    #     return ''
    #  parts = []
    #  for k, v in d.items():
    #     if isinstance(v, list) and v:
    #         parts.append(f"{k}:[{'; '.join(v)}]")
    #     else:
    #         parts.append(f"{k}:[]")
    #     return ', '.join(parts)

    # df['skill_details_str'] = df['skill_details'].apply(pretty_details)
    # return df   

def parse_salary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Vectorized salary extraction from extensions + description.
    NA-safe, handles ranges, k-suffix, units, EU thousand dots.
    Produces: salary_min_annual, salary_max_annual, salary_mid_annual, salary_currency.
    """
    df = df.copy()

    # Unified salary text
    ext_s  = df['extensions'].fillna('').astype(str).str.lower()
    desc_s = df['description'].fillna('').astype(str).str.lower()
    sal_s  = ext_s + ' ' + desc_s

    pat = re.compile(
        r'(?P<cur>(?:usd|cad|ca\$|us\$|c\$|€|£|\$)?)\s*'
        r'(?P<a1>\d{2,6}(?:[.,]\d{3})*(?:\.\d+)?)\s*(?P<a1k>[kK])?'
        r'(?:\s*(?:\-|–|to|—)\s*'
        r'(?P<a2>\d{2,6}(?:[.,]\d{3})*(?:\.\d+)?)\s*(?P<a2k>[kK])?)?'
        r'\s*(?:per\s*)?(?P<unit>year|yr|annum|hour|hr|day|week|wk|month|mo)?',
        flags=re.IGNORECASE
    )

    matches = sal_s.str.extractall(pat)

    # plausibility: currency OR unit OR 'k'
    has_k = matches[['a1k','a2k']].apply(lambda s: s.fillna('').astype(str).str.len() > 0).any(axis=1)
    has_signal = (
        matches['cur'].fillna('').astype(str).ne('') |
        matches['unit'].notna() |
        has_k
    )
    m = matches[has_signal].copy()
    if m.empty:
        # make sure columns exist even if empty
        for c in ['salary_min_annual','salary_max_annual','salary_mid_annual','salary_currency']:
            if c not in df.columns:
                df[c] = np.nan
        return df

    # number cleanup
    def clean_num(s: pd.Series) -> pd.Series:
        s = s.fillna('').astype(str)
        s = s.str.replace(',', '', regex=False)
        s = s.str.replace(r'(?<=\d)\.(?=\d{3}\b)', '', regex=True)
        s = s.str.replace(r'(?<=\d)\.(?=\d{3}(?:\D|$))', '', regex=True)
        return s

    m['n1'] = clean_num(m['a1'])
    m['n2'] = clean_num(m['a2'].fillna(m['a1']))
    m['v1'] = pd.to_numeric(m['n1'], errors='coerce')
    m['v2'] = pd.to_numeric(m['n2'], errors='coerce')

    a1k_flag = m['a1k'].fillna('').astype(str).str.lower().eq('k')
    a2k_flag = m['a2k'].fillna('').astype(str).str.lower().eq('k')
    m['v1'] = np.where(a1k_flag, m['v1']*1000, m['v1'])
    m['v2'] = np.where(a2k_flag, m['v2']*1000, m['v2'])

    unit = m['unit'].fillna('').astype(str).str.lower().replace({'': 'year'})
    factor_map = {'year':1,'yr':1,'annum':1,'hour':2080,'hr':2080,'day':260,'week':52,'wk':52,'month':12,'mo':12}
    m['factor'] = unit.map(factor_map).fillna(1)

    m['annual_min'] = m['v1'] * m['factor']
    m['annual_max'] = m['v2'] * m['factor']

    cur_raw = m['cur'].fillna('').astype(str).str.lower()
    m['currency'] = np.select(
        [
            cur_raw.str.contains(r'cad|ca\$|c\$', regex=True),
            cur_raw.str.contains(r'usd|us\$', regex=True),
            cur_raw.str.contains(r'€'),
            cur_raw.str.contains(r'£'),
            cur_raw.str.contains(r'\$')
        ],
        ['CAD','USD','EUR','GBP','USD'],
        default=''
    )

    # infer currency from location if still blank
    m_idx = m.index.get_level_values(0)
    loc_aligned = (df.reindex(m_idx)['location_bucket'].fillna('').astype(str)
                   if 'location_bucket' in df.columns else pd.Series('', index=m_idx))
    cur_fill = np.where(loc_aligned.eq('Canada'), 'CAD',
                 np.where(loc_aligned.eq('United States'), 'USD', 'USD'))
    m['currency'] = m['currency'].astype(str).mask(m['currency'].eq(''), cur_fill)

    # guardrails
    ok = (m['annual_min'].between(10_000, 500_000)) & (m['annual_max'].between(10_000, 500_000))
    m = m[ok]
    if m.empty:
        for c in ['salary_min_annual','salary_max_annual','salary_mid_annual','salary_currency']:
            if c not in df.columns:
                df[c] = np.nan
        return df

    # aggregate
    agg = (m.groupby(level=0)
             .agg(salary_min_annual=('annual_min','min'),
                  salary_max_annual=('annual_max','max'),
                  salary_currency=('currency','first')))

    # KEY FIX: drop existing salary cols before join to avoid overlap
    overlap = [c for c in ['salary_min_annual','salary_max_annual','salary_currency'] if c in df.columns]
    if overlap:
        df = df.drop(columns=overlap)

    df = df.join(agg)
    df['salary_mid_annual'] = df[['salary_min_annual','salary_max_annual']].mean(axis=1)

    return df

def label_comp_type_and_guard_salaries(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    desc  = df['description'].fillna('').str.lower()
    ext   = df['extensions'].fillna('').astype(str).str.lower()
    via   = df['via'].fillna('').str.lower()
    title = df['title'].fillna('').str.lower()

    is_upwork   = via.str.contains('upwork')
    is_contract = ext.str.contains(r'\bcontract|contractor\b') | title.str.contains(r'\bcontract\b', regex=True)
    is_hourly   = ext.str.contains(r'\b/hour|per hour|hourly\b') | desc.str.contains(r'\b/hour|per hour|hourly\b', regex=True)
    is_project  = desc.str.contains(r'\bper\s*project|per\s*task|project[-\s]?based\b', regex=True)

    df['comp_type'] = np.select(
        [is_upwork | is_project, is_hourly, is_contract],
        ['project', 'hourly', 'contract'],
        default='salary'
    )

    mask_bad = (df['comp_type'] != 'salary') | (~df['salary_mid_annual'].between(20_000, 500_000))
    df.loc[mask_bad, ['salary_min_annual','salary_max_annual','salary_mid_annual','salary_currency']] = np.nan
    return df

# ---------- ONE-SHOT PIPE FOR A SINGLE DF ----------
def prepare_jobs_df(raw: pd.DataFrame) -> pd.DataFrame:
    """Normalize + enrich a single dataframe into the master schema."""
    df = coerce_schema(raw)
    # Enrich in safe order
    df = add_job_basket(df)
    df = add_location_bucket(df)
    df = add_seniority(df)
    df = extract_skills(df)
    df = parse_salary(df)
    df = label_comp_type_and_guard_salaries(df)
    if ONLY_DATA_ROLES:
        df = df[df['job_basket'] != 'Other'].copy()
    df.reset_index(drop=True, inplace=True)
    return df

# ---------- MAIN: load multiple CSVs, normalize, enrich, concat ----------
def build_master(file_paths, out_path=None):
    frames = []
    for fp in file_paths:
        fp = Path(fp)
        df = pd.read_csv(fp, low_memory=False)

        # route by filename (simple & robust)
        if fp.name.lower().startswith('jobs_2024'):
            norm = normalize_2024(df, fp.name)
        else:
            norm = normalize_gsearch_q(df, fp.name)

        enriched = prepare_jobs_df(norm)
        enriched['source_file'] = fp.name
        enriched['listed_year'] = pd.to_numeric(enriched['listed_time'], errors='coerce').astype('Int64')
        frames.append(enriched)

    master = pd.concat(frames, ignore_index=True)

    # de-dupe 
    dedupe_keys = ['title','company_name','job_location','via','description','listed_time']
    master = master.drop_duplicates(subset=dedupe_keys, keep='first')

    if out_path:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        master.to_csv(out_path, index=False)
    return master

# ---------- EXAMPLE USAGE ----------
# Update base_dir to your folder path, then run.
base_dir = r"/Users/pardeepwalia/Desktop/untitled folder/Historical data set "
files = [
    f"{base_dir}/gsearch_jobs_2023_q1.csv",
    f"{base_dir}/gsearch_jobs_2023_q2.csv",
    f"{base_dir}/gsearch_jobs_2023_q3.csv",
    f"{base_dir}/jobs_2024.csv",
    f"{base_dir}/gsearch_jobs_2022.csv",
]

master = build_master(files, out_path=f"{base_dir}/jobs_master_2023_2024_wrangled.csv")
print("Master shape:", master.shape)
print("Columns:", list(master.columns))
print(master.head(3))

/var/folders/sl/2tbdr8850qq877l7gbhpjzcm0000gn/T/ipykernel_38099/4139517016.py:111: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  lt = pd.to_datetime(df.get('posted_at', pd.NaT), errors='coerce')
/var/folders/sl/2tbdr8850qq877l7gbhpjzcm0000gn/T/ipykernel_38099/4139517016.py:113: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  lt = pd.to_datetime(df.get('date_time', pd.NaT), errors='coerce')
/var/folders/sl/2tbdr8850qq877l7gbhpjzcm0000gn/T/ipykernel_38099/4139517016.py:111: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  lt = pd.to_datetime(df.get('posted_at', pd.NaT), errors='c

Master shape: (19749, 30)
Columns: ['title', 'company_name', 'job_location', 'via', 'description', 'extensions', 'description_tokens', 'job_basket', 'listed_time', 'location_bucket', 'seniority_level', 'salary_mid_annual', 'SQL', 'Python', 'Excel', 'Tableau', 'Power BI', 'Cloud', 'Cloud_list', 'skill_buckets', 'skill_buckets_str', 'skill_details', 'skill_details_str', 'skills_flat', 'salary_min_annual', 'salary_max_annual', 'salary_currency', 'comp_type', 'source_file', 'listed_year']
                                               title     company_name  \
0                                    Data Analyst Sr        CalOptima   
1                Data Analyst, Space Operations Data  Virgin Galactic   
2  DATA ANALYST  We offer career advancement a wo...   Zunch Staffing   

    job_location                          via  \
0  United States                    via WayUp   
1  United States  via Virgin Galactic Careers   
2  United States                    via WayUp   

                    

In [2]:
# updating the listed year
master['listed_year'] = master['source_file'].str.extract(r'(\d{4})').astype(int)

In [3]:
master.columns

Index(['title', 'company_name', 'job_location', 'via', 'description',
       'extensions', 'description_tokens', 'job_basket', 'listed_time',
       'location_bucket', 'seniority_level', 'salary_mid_annual', 'SQL',
       'Python', 'Excel', 'Tableau', 'Power BI', 'Cloud', 'Cloud_list',
       'skill_buckets', 'skill_buckets_str', 'skill_details',
       'skill_details_str', 'skills_flat', 'salary_min_annual',
       'salary_max_annual', 'salary_currency', 'comp_type', 'source_file',
       'listed_year'],
      dtype='object')

In [4]:
master['location_bucket']

0        United States
1        United States
2               Remote
3        United States
4               Remote
             ...      
23919    United States
23922           Remote
23923    United States
23924    United States
23925    United States
Name: location_bucket, Length: 19749, dtype: object

In [5]:
master.to_csv(r"/Users/pardeepwalia/Desktop/untitled folder/Historical data set /jobs_master_2023_2024_wrangled.csv",index=False)